# Group Relative Policy Optimization (GRPO) with veRL on Amazon SageMaker Training jobs
## Lab 1 - Data preparation

**Stage: Data.** In this notebook we prepare the [GSM8K](https://huggingface.co/datasets/openai/gsm8k) dataset in the schema that [veRL](https://github.com/volcengine/verl) expects for reinforcement learning, and upload it to Amazon S3 for the training job in Lab 2.

### What you will do

| Step | What happens | Input | Output |
| --- | --- | --- | --- |
| 1 | Install the notebook's Python dependencies | `requirements.txt` | Packages in the kernel |
| 2 | Resolve the SageMaker session, execution role, and default bucket | Your Studio environment | `role`, `bucket_name`, `s3_client` |
| 3 | Load GSM8K from the Hugging Face Hub and look at a record | `openai/gsm8k` | `gsm8k`, a `DatasetDict` with `train` and `test` splits |
| 4 | Fix the two constants that drive scoring, and extract a ground truth | A GSM8K `answer` string | `DATA_SOURCE`, `INSTRUCTION`, `extract_ground_truth()` |
| 5 | Map GSM8K records into veRL's five-column schema and subsample | `gsm8k["train"]`, `gsm8k["test"]` | `train_dataset` (1,280 rows), `validation_dataset` (256 rows) |
| 6 | Write both splits to parquet and upload them to S3 | The two datasets | Two S3 URIs, plus `./tmp/gsm8k_eval.jsonl` for Lab 4 |

Nothing here needs a GPU, and it takes about ten minutes. The theory behind the schema -- why an RL dataset carries a *check* rather than an *answer* -- is on the workshop's Lab 1 page; this notebook concentrates on doing it.

## Prerequisites

### Step 1 - Install requirements

**What this does:** installs the notebook's Python dependencies into the kernel.

**Input:** `./requirements.txt`. **Output:** pip's install log. A note that you may need to restart the kernel is normal; ignore it unless an import below fails.

In [ ]:
%pip install -r ./requirements.txt --upgrade

### Step 2 - Resolve the SageMaker session

**What this does:** creates a SageMaker session and reads the three things every later step uses: the execution role this notebook runs as, the account's default SageMaker bucket, and the region.

**Input:** nothing -- all three come from the Studio environment. **Output:** three printed lines. The role ARN should contain `SageMakerExecutionRole`, the bucket should be `sagemaker-<region>-<account id>`, and the region should match your event. If the bucket did not exist yet, the SDK creates it and logs that it did.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")

sess = Session(default_bucket=sagemaker_session_bucket)

bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## Step 3 - Load and look at the dataset

**What this does:** downloads GSM8K from the Hugging Face Hub and shows what a record looks like.

**Input:** the dataset id `openai/gsm8k`, configuration `main`. **Output:** a `DatasetDict` with a `train` split of 7,473 rows and a `test` split of 1,319 rows; each row is a `question` and an `answer`. The second cell shows the first five rows as a table.

In this example, we are going to load [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k), 8.5k grade-school maths word problems. Each problem has a worked solution that ends with the final answer after a `####` marker.

That marker is the reason this dataset suits GRPO. Supervised fine-tuning and DPO both learn from text we hand them -- a target completion, or a preferred and a rejected one. GRPO learns from text the model writes itself: for every prompt it samples a group of completions, scores each one, and pushes the policy toward the ones that scored well. So the dataset does not need a good answer to imitate. It needs a **checkable** answer to score against.

In [ ]:
import datasets
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
gsm8k

In [ ]:
import pandas as pd

df = pd.DataFrame(gsm8k["train"])
df.head()

A single problem in full, so the `####` convention is visible.

**Expected output:** the question, a blank line, then a worked solution whose last line is `#### 72`. That last line is the only part of the solution the reward function will ever look at.

In [ ]:
print(df.loc[0, "question"])
print()
print(df.loc[0, "answer"])

## The veRL reinforcement-learning schema
veRL reads its data as parquet with five columns, and each one has a job:

| column | what it carries |
| --- | --- |
| `data_source` | Which reward function scores this row |
| `prompt` | The chat messages the model is asked to complete |
| `ability` | A free-text tag, for filtering and reporting |
| `reward_model` | The scoring style, and the ground truth to score against |
| `extra_info` | Split name, row index, and anything else we want to keep |

`data_source` is the field worth pausing on, because it is doing something less obvious than it looks. We are not writing a reward function in this lab and we are not passing a flag to select one. veRL's default reward manager reads `data_source` off each row and dispatches on its value, so the literal string `openai/gsm8k` is what routes every row to veRL's built-in GSM8K scorer at `verl/utils/reward_score/gsm8k.py`. That scorer greps the model's completion for `#### <number>` and compares it to `reward_model.ground_truth`. Change the string and scoring silently stops working, so we set it once, in one place, below.

The second thing that scorer implies is a prompt requirement. It can only find an answer the model actually formats with `####`, so we append an instruction telling the model to do that. Without it the completions may well be correct and will still score zero.

### Step 4 - The two constants that drive scoring

**What this does:** fixes `DATA_SOURCE` -- the string that selects veRL's GSM8K scorer -- and `INSTRUCTION`, the sentence appended to every prompt so the model produces a `####` answer the scorer can find. It also defines `extract_ground_truth()`, which pulls the final number out of a GSM8K solution.

**Input:** a GSM8K `answer` string. **Output:** the ground truth as a plain string with commas removed -- `72` for the first training record. That string is what veRL's scorer will compare the model's `####` answer against during training.

In [ ]:
import re

DATA_SOURCE = "openai/gsm8k"
INSTRUCTION = 'Let\'s think step by step and output the final answer after "####".'


def extract_ground_truth(solution: str) -> str:
    """Pull the final answer out of a GSM8K solution.

    The solutions end with `#### <number>`. We keep the number only, with commas
    removed, because that is the form veRL's scorer compares against.
    """
    match = re.search(r"#### (\-?[0-9\.\,]+)", solution)
    if match is None:
        raise ValueError(f"no #### answer found in: {solution!r}")
    return match.group(0).split("#### ")[1].replace(",", "")


print(extract_ground_truth(df.loc[0, "answer"]))

### Step 5 - Map records into the veRL schema

**What this does:** defines `to_verl_row()`, which turns one GSM8K record into one row of the five-column schema above, and `build_split()`, which applies it to a whole split and optionally shuffles and truncates it to a fixed number of rows.

**Input:** a `question`, an `answer`, the split name, and the row index. **Output:** a Python dict with the five keys. Notice what is *not* in it: there is no target completion. `prompt` is the question plus the instruction, and `reward_model.ground_truth` is the number to check against. The original worked solution is kept only under `extra_info`, for reference; veRL does not train on it.

In [ ]:
def to_verl_row(question: str, answer: str, split: str, index: int) -> dict:
    return {
        "data_source": DATA_SOURCE,
        "prompt": [{"role": "user", "content": f"{question} {INSTRUCTION}"}],
        "ability": "math",
        "reward_model": {
            "style": "rule",
            "ground_truth": extract_ground_truth(answer),
        },
        "extra_info": {
            "split": split,
            "index": index,
            "answer": answer,
            "question": question,
        },
    }


def build_split(hf_split, split_name: str, rows: int | None, seed: int = 42):
    prepared = [
        to_verl_row(record["question"], record["answer"], split_name, index)
        for index, record in enumerate(hf_split)
    ]
    prepared_ds = datasets.Dataset.from_list(prepared)
    if rows is not None and rows < len(prepared_ds):
        prepared_ds = prepared_ds.shuffle(seed=seed).select(range(rows))
    return prepared_ds

We are deliberately not training on all 7473 rows. GRPO samples `rollout_n` completions for every prompt in a step, so a step is far more expensive than a supervised one -- on `ml.g6e.12xlarge` it takes about ten minutes. 1280 rows at a batch size of 128 is exactly 10 steps, which is enough to move the policy measurably and short enough to finish inside a lab.

Validation is trimmed for the same reason: veRL generates a completion for every validation row, and it does so twice, once before training starts and once at the end. Those two passes are what let us say whether the ten steps changed anything, so we keep them cheap.

**Expected output** of the next cell: `train rows: 1280` and `validation rows: 256`. Lab 1's page asks you to confirm exactly these two numbers before moving on.

In [ ]:
# GSM8K ships train and test; veRL validates on what we mount as the
# validation channel, so the test split becomes validation here.
train_dataset = build_split(gsm8k["train"], "train", rows=1280)
validation_dataset = build_split(gsm8k["test"], "test", rows=256)

print(f"train rows:      {len(train_dataset)}")
print(f"validation rows: {len(validation_dataset)}")

Inspect one prepared row, which is what veRL will actually read.

**Expected output:** one row printed as a nested dict. Find the instruction at the end of `prompt[0]['content']`, and the `ground_truth` under `reward_model`. Those two fields are the whole contract with the scorer: the first makes the model write a `####` line, the second is what that line is compared against.

In [ ]:
import random
from rich.pretty import pprint

pprint(train_dataset[random.randint(0, len(train_dataset) - 1)])

### Step 6 - Upload to Amazon S3

**What this does:** writes each split to a parquet file, uploads both to the account's default SageMaker bucket under `datasets/llm-fine-tuning-grpo/`, and saves the validation questions locally for Lab 4.

**Input:** `train_dataset` and `validation_dataset`. **Output:** two S3 URIs printed at the end, ending in `train/dataset.parquet` and `validation/dataset.parquet`. Lab 2 re-derives these same paths from the session rather than reading them from here, so there is nothing to copy.

veRL loads a *file*, not a directory, so each channel path names the parquet file directly. We also keep a copy of the validation questions on local disk, which Lab 4 reads to evaluate the trained model.

In [ ]:
import os
import shutil

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/validation", exist_ok=True)
os.makedirs("./tmp", exist_ok=True)

train_dataset.to_parquet("./data/train/dataset.parquet")
validation_dataset.to_parquet("./data/validation/dataset.parquet")

if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-grpo"
else:
    input_path = "datasets/llm-fine-tuning-grpo"

s3_client.upload_file(
    "./data/train/dataset.parquet", bucket_name, f"{input_path}/train/dataset.parquet"
)
s3_client.upload_file(
    "./data/validation/dataset.parquet",
    bucket_name,
    f"{input_path}/validation/dataset.parquet",
)

# Lab 4 evaluates against the same questions veRL validated on.
validation_dataset.to_json("./tmp/gsm8k_eval.jsonl")

shutil.rmtree("./data")

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.parquet"
validation_dataset_s3_path = (
    f"s3://{bucket_name}/{input_path}/validation/dataset.parquet"
)

print(f"train: {train_dataset_s3_path}")
print(f"validation: {validation_dataset_s3_path}")